# Numpy를 사용하여 RNN 동작 이해

In [19]:
import numpy as np

# RNN이 몇 번 펼쳐질 것인가를 의미하는 숫자. 일반적으로는 단어의 개수. 즉 문장의 길이
timestemps = 10 # 문장의 길이 t

# RNN의 입력. 일반적으로는 단어의 벡터 차원
input_size = 4 # 단어의 차원 D

# RNN 셀에서의 히든 유닛의 개수. 이것을 셀의 용량이라고 한다.
hidden_size = 8

In [20]:
# RNN 입력 데이터 ( projection layer )
inputs = np.random.random((timestemps, input_size)) # D x t 형태의 행렬

# hidden state
hidden_state_t = np.zeros((hidden_size, ))

In [21]:
# 문장 행렬
#  단어 벡터의 차원은 4차원.
#  문장의 길이는 10으로 가정
inputs

array([[0.98277694, 0.67386033, 0.86091488, 0.51990459],
       [0.17716463, 0.63805084, 0.69782566, 0.46636883],
       [0.48403916, 0.94175796, 0.91701299, 0.10496689],
       [0.39608918, 0.69731486, 0.65942339, 0.0272592 ],
       [0.70864417, 0.81763889, 0.08188353, 0.66326869],
       [0.3946444 , 0.49236533, 0.42087498, 0.33230011],
       [0.42524011, 0.25421465, 0.278766  , 0.78133729],
       [0.0103837 , 0.22681407, 0.45048012, 0.05933932],
       [0.98383508, 0.59198864, 0.78516913, 0.04376862],
       [0.07358311, 0.04266258, 0.37348016, 0.3910711 ]])

In [22]:
print(hidden_state_t)

[0. 0. 0. 0. 0. 0. 0. 0.]


In [23]:
# RNN Cell의 뉴런 가중치 설정
    # 1. 입력 x_t와 대응되는 가중치 ( D x hidden_size )
    # 2. 이전 시점의 상태인 h_t-1에 대응되는 가중치 ( hidden_size X hidden_size )

Wx = np.random.random((hidden_size, input_size)) # ( 8 x 4 )
Wh = np.random.random((hidden_size, hidden_size)) # ( 8 x 8 )
b = np.random.random((hidden_size,)) # (8, )

$$
h_t = tanh(W_xx_t + W_hh_{t-1} + b)
$$

In [24]:
total_hidden_states = []

# RNN 작동

# 단어 벡터를 하나씩 순서대로 꺼낸다.
for input_t in inputs:
    # output_t가 실제로는 h_t(현 시점의 hidden state)
    output_t = np.tanh(Wx@input_t + Wh@hidden_state_t + b) # 위 식을 코드로 적은 것

    # 각 시점의 은닉 상태의 값을 계속해서 기록
    total_hidden_states.append(list(output_t))

    hidden_state_t = output_t

# 출력 시 값을 깔끔하게 해줌
total_hidden_states = np.stack(total_hidden_states, axis=0)

total_hidden_states

array([[0.90368183, 0.87427087, 0.98152143, 0.94738759, 0.92194694,
        0.98599854, 0.69524324, 0.65835662],
       [0.9999254 , 0.99980783, 0.99993813, 0.99981547, 0.99989769,
        0.99995894, 0.99756752, 0.99977277],
       [0.9999774 , 0.99994011, 0.99999076, 0.99990481, 0.99997338,
        0.99999496, 0.99935935, 0.99995117],
       [0.99995829, 0.99990208, 0.99997458, 0.99985555, 0.99995747,
        0.99998941, 0.99893728, 0.99993702],
       [0.99996515, 0.99996769, 0.99996824, 0.99996054, 0.99998392,
        0.99999467, 0.99913479, 0.99995387],
       [0.99995757, 0.99992039, 0.9999563 , 0.99990196, 0.99996656,
        0.99998488, 0.99863022, 0.99992916],
       [0.99996971, 0.99995061, 0.99994291, 0.99995068, 0.99998102,
        0.9999807 , 0.99836314, 0.99992217],
       [0.99991895, 0.99978536, 0.9999008 , 0.99976042, 0.99991189,
        0.99994946, 0.99770337, 0.99989021],
       [0.99997799, 0.99994874, 0.9999832 , 0.99989667, 0.99998316,
        0.99999564, 0.998929

# PyTorch를 이용한 RNN

## Basic RNN

In [25]:
import torch
import torch.nn as nn

In [26]:
input_size = 5 # 입력되는 단어 벡터의 차원
hidden_size = 8 # 셀의 용량

`[I, am, a , Student]`라는 문장이 들어간다고 가정

In [27]:
# 파이토치에 넣을 때는 구성이 다음과 같아야 한다: 
# (batch_size, timesteps, input_size) ==> (배치 크기, 문장 길이, 단어 벡터 크기)

inputs = torch.Tensor(1, 4, input_size)
inputs


tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]])

In [28]:
# (데이터의 개수 / 배치 크기, 문장의 길이, 단어 벡터의 차원)
inputs.shape

torch.Size([1, 4, 5])

In [30]:
# RNN 레이어에 필요한 건? 입력 데이터의 차원, 뉴런의 개수

cell = nn.RNN(input_size, hidden_size, batch_first=True) 
# 5차원 데이터가 4번(4개 X) 들어감. 그리고 이를 받아내는 뉴런은 8개
# 다만 들어가는 횟수는 적어주지 않아도 됨 -> 가변적이기 때문

In [32]:
outputs, hidden = cell(inputs) # inputs : (1, 4, 5) 뉴런은 (5, 8)

In [33]:
# 모든 timestep 각각의 hidden state
outputs

tensor([[[ 0.1470,  0.2153, -0.0855,  0.3412,  0.0982, -0.4084,  0.2391,
           0.4778],
         [ 0.2929,  0.4187, -0.4007,  0.2175,  0.2559, -0.3308,  0.5505,
           0.5723],
         [ 0.3770,  0.4951, -0.4707,  0.2312,  0.3154, -0.3977,  0.5879,
           0.6529],
         [ 0.4253,  0.5331, -0.5067,  0.2232,  0.3765, -0.3887,  0.6271,
           0.6673]]], grad_fn=<TransposeBackward1>)

In [34]:
# 제일 마지막 timestep의 hidden state
hidden

tensor([[[ 0.4253,  0.5331, -0.5067,  0.2232,  0.3765, -0.3887,  0.6271,
           0.6673]]], grad_fn=<StackBackward0>)

Many to one과 Many to many

- Many to One은 여러 timestep을 입력 받아 하나의 결과를 낸다.
- Many to Many는 여러 timestep을 입력 받아 여러 결과를 낸다.

## Deep RNN (Deep Recurrent Neural Network)
RNN 층이 여러 겹으로 쌓여있는 경우

In [36]:
# ( batch_size, timestep, input_size )

inputs = torch.Tensor(1, 4, 5)

In [37]:
cell = nn.RNN(input_size=5, hidden_size=8, batch_first=True, num_layers=2) # RNN 층을 2층으로
outputs, hidden = cell(inputs)


In [38]:
# outputs는 모든 타임 스텝 각각의 hidden state이므로 층이 몇 층이든 똑같다.
outputs.shape

torch.Size([1, 4, 8])

In [39]:
# 각 층의 마지막 timestep ( 층 수, 배치 크기, hidden state )
hidden.shape

torch.Size([2, 1, 8])

## Bi-RNN (Bidirectional RNN)
- 양방향 RNN

In [40]:
inputs = torch.Tensor(1, 4, 5)

In [41]:
cell = nn.RNN(input_size=5, hidden_size=8, batch_first=True, bidirectional=True)
outputs, hidden = cell(inputs)

In [43]:
# (1, 4, 16) 인 이유는? 순 방향 hidden state 8개, 역 방향 hidden state 8개 하니까 하나의 셀에 16개의 hidden state
outputs.shape

torch.Size([1, 4, 16])

In [44]:
# 0 번째가 순 방향 hidden state
# 1 번째가 역 방향 hidden state

hidden.shape # 순방향의 마지막 hidden state와 역 방향의 마지막 hidden

torch.Size([2, 1, 8])

In [45]:
# 순 방향 hidden state
hidden[0]

tensor([[-0.4504, -0.1935, -0.2883, -0.0474, -0.0767,  0.0735,  0.0251,  0.2387]],
       grad_fn=<SelectBackward0>)

In [46]:
# 역 방향 hidden state
hidden[1]

tensor([[-0.0815,  0.0335,  0.1671,  0.1536, -0.5015, -0.2592,  0.3675, -0.2805]],
       grad_fn=<SelectBackward0>)